In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB

# ==========================================
# 1. Carregamento e Preparação dos Dados
# ==========================================
df = pd.read_csv('pet_adoption_data.csv')

# Remoção do ID do pet antigo
if 'PetID' in df.columns:
    df = df.drop(columns=['PetID'])

# Criamos um dicionário para guardar os LabelEncoders de cada coluna
label_encoders = {}
colunas_categoricas = df.select_dtypes(include=['object']).columns

for col in colunas_categoricas:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Salva o encoder treinado

# Divisão dos dados em Recursos (X) e Alvo (y)
X = df.drop(columns=['AdoptionLikelihood'])
y = df['AdoptionLikelihood']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Padronização (Ajustada apenas com os dados de treino)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Treinamento do Modelo Naive Bayes Gaussiano
modelo = GaussianNB()
modelo.fit(X_train_scaled, y_train)

# ==========================================
# 2. Definição de Apenas UM Novo Pet
# ==========================================
# Aqui define as características exatas do pet que quer testar:
dados_do_pet = {
    'PetType': 'Dog',
    'Breed': 'Golden Retriever',
    'AgeMonths': 3,               # Um filhote de 3 meses
    'Color': 'Orange',
    'Size': 'Medium',
    'WeightKg': 8.5,              # Peso em Kg
    'Vaccinated': 1,              # 1 = Sim, 0 = Não
    'HealthCondition': 0,         # 0 = Saudável
    'TimeInShelterDays': 5,       # Está há apenas 5 dias no abrigo
    'AdoptionFee': 150,           # Taxa de adoção
    'PreviousOwner': 0            # 0 = Nunca teve dono anterior
}

# Transforma o dicionário em um DataFrame de uma linha
df_novo_pet = pd.DataFrame([dados_do_pet])

print("--- Dados do Pet para Análise ---")
print(df_novo_pet.to_string(index=False))

# ==========================================
# 3. Processamento e Previsão
# ==========================================
# Criamos uma cópia para aplicar as transformações matemáticas
df_novo_pet_processado = df_novo_pet.copy()

# 1º Passo: Transformar o texto em número usando os mesmos encoders do treino
for col in colunas_categoricas:
    le = label_encoders[col]
    df_novo_pet_processado[col] = le.transform(df_novo_pet_processado[col])

# 2º Passo: Padronizar os números usando o mesmo scaler do treino
novo_pet_scaled = scaler.transform(df_novo_pet_processado)

# 3º Passo: O modelo faz a previsão
previsao = modelo.predict(novo_pet_scaled)
probabilidade = modelo.predict_proba(novo_pet_scaled)

print("\n--- Resultado da Previsão ---")
if previsao[0] == 1:
    print("Resultado: Este pet tem uma ALTA probabilidade de ser adotado! 🐶")
else:
    print("Resultado: Este pet tem uma BAIXA probabilidade de ser adotado. 🐾")

# Mostra a probabilidade exata calculada pelo Naive Bayes
chance_adocao = probabilidade[0][1] * 100
print(f"Probabilidade exata calculada pelo modelo: {chance_adocao:.2f}%")

--- Primeiras linhas do dataset ---
   PetID PetType             Breed  AgeMonths   Color    Size   WeightKg  \
0    500    Bird          Parakeet        131  Orange   Large   5.039768   
1    501  Rabbit            Rabbit         73   White   Large  16.086727   
2    502     Dog  Golden Retriever        136  Orange  Medium   2.076286   
3    503    Bird          Parakeet         97   White   Small   3.339423   
4    504  Rabbit            Rabbit        123    Gray   Large  20.498100   

   Vaccinated  HealthCondition  TimeInShelterDays  AdoptionFee  PreviousOwner  \
0           1                0                 27          140              0   
1           0                0                  8          235              0   
2           0                0                 85          385              0   
3           0                0                 61          217              1   
4           0                0                 28           14              1   

   AdoptionLikelihoo